# omim phenotypic series enrichment

In [1]:
"""
analyze_omim_diseases_enrichr.py - Analyze OMIM disease-gene associations using Enrichr API

Usage:
    python analyze_omim_diseases_enrichr.py --ppi_source STRING --input_dir ./input --output_dir ./output
"""

import argparse
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
from scipy.stats import fisher_exact
from statsmodels.stats.multitest import multipletests
from collections import defaultdict
import os
import requests
import pickle
import json
import time
from common_utils import *
from hidef_multiplex_adapter import MultiplexClusterLoaderMixin

In [2]:
class OMIMPhenotypicSeriesAnalyzer(MultiplexClusterLoaderMixin):
    def __init__(self, ppi_source, input_dir, output_dir, omim_ps_file, persistence_threshold):
        import pipeline_config as cfg
        if persistence_threshold is None:
            persistence_threshold = cfg.PERSISTENCE_THRESHOLD
        print(f"\n{'='*80}")
        print(f"INITIALIZING OMIMPhenotypicSeriesAnalyzer")
        print(f"{'='*80}")
        print(f"PPI Source: {ppi_source}")
        print(f"Input Dir: {input_dir}")
        print(f"Output Dir: {output_dir}")
        print(f"OMIM PS File: {omim_ps_file}")
        
        self.ppi_source = ppi_source
        self.input_dir = input_dir
        self.output_dir = output_dir
        self.omim_ps_file = omim_ps_file
        self.persistence_threshold = persistence_threshold
        
        # Load OMIM phenotypic series data
        print(f"\nLoading OMIM Phenotypic Series data...")
        try:
            self.omim_ps_df = pd.read_csv(omim_ps_file)
            print(f"  ✓ Loaded {len(self.omim_ps_df)} phenotypic series entries")
            print(f"  Columns: {list(self.omim_ps_df.columns)}")
        except Exception as e:
            print(f"  ✗ Error loading OMIM PS file: {e}")
            raise
       
        # Setup output directories
        self.cluster_type = 'hidef_from_comparison'
        self.analysis_dir = os.path.join(output_dir, ppi_source, f'omim_ps_analysis_{self.cluster_type}')
           
        print(f"\nOutput Configuration:")
        print(f"  Analysis directory: {self.analysis_dir}")
        
        ensure_dir(self.analysis_dir)
        print(f"  Directory created/verified: {os.path.exists(self.analysis_dir)}")
        
        # Data containers
        self.clusters = None
        self.seed_genes = None
        self.cluster_seed_mapping = None
        self.results = []
        self.gene_results = []
        
        print(f"{'='*80}\n")

    @property
    def safe_prefix(self):
        """Return a safe prefix for filename generation"""
        return self.cluster_type
    
    def safe_log2(self, value, min_value=1e-10, max_value=1e10):
        """Safely calculate log2 with handling for zero/negative values and infinite results."""
        bounded_value = max(min_value, min(value, max_value))
        return np.log2(bounded_value)
    
    def calculate_overlap_pvalue(self, overlap_count, cluster_size, ps_size, total_genes=20000):
        """Calculate hypergeometric p-value for overlap between cluster and phenotypic series."""
        from scipy.stats import hypergeom
        
        # Hypergeometric test: probability of observing overlap_count or more overlaps
        # by chance when drawing cluster_size genes from total_genes, where ps_size are "success" genes
        pval = hypergeom.sf(overlap_count - 1, total_genes, ps_size, cluster_size)
        return pval
    
    def analyze_enrichment(self):
        """Analyze phenotypic series enrichment in each cluster."""
        print(f"\n{'='*80}")
        print(f"STARTING PHENOTYPIC SERIES ENRICHMENT ANALYSIS")
        print(f"{'='*80}")

        if not self.clusters:
            print("ERROR: No clusters loaded!")
            return pd.DataFrame(), pd.DataFrame()

        print(f"Total clusters to process: {len(self.clusters)}")
        print(f"Total phenotypic series: {self.omim_ps_df['ps_code'].nunique()}")

        summarized_results = []

        for cluster_idx, (cluster_name, cluster_genes) in enumerate(self.clusters.items()):
            print(f"\n{'-'*80}")
            print(f"Processing cluster {cluster_idx + 1}/{len(self.clusters)}: {cluster_name}")
            print(f"  Cluster size: {len(cluster_genes)} genes")

            cluster_genes_list = [str(g).upper() for g in cluster_genes]
            cluster_genes_set = set(cluster_genes_list)
            cluster_seed_genes = self.cluster_seed_mapping[cluster_name]
            print(f"  Seed genes: {len(cluster_seed_genes)}")

            # Get unique phenotypic series
            ps_codes = self.omim_ps_df['ps_code'].unique()
            results_added = 0

            for ps_code in ps_codes:
                ps_data = self.omim_ps_df[self.omim_ps_df['ps_code'] == ps_code]
                ps_title = ps_data['ps_title'].iloc[0] if len(ps_data) > 0 else ps_code

                # Get all genes in this phenotypic series
                ps_genes_str = ps_data['genes'].str.upper().str.split(',').explode().str.strip()
                ps_genes_set = set(ps_genes_str.unique())

                # Calculate overlap
                overlapping_genes = cluster_genes_set.intersection(ps_genes_set)
                overlap_count = len(overlapping_genes)

                if overlap_count == 0:
                    continue

                # Calculate statistics
                percentage = (overlap_count / len(cluster_genes_list)) * 100

                # Calculate p-value using hypergeometric test
                p_value = self.calculate_overlap_pvalue(
                    overlap_count, 
                    len(cluster_genes_list), 
                    len(ps_genes_set)
                )

                # Placeholder - will be FDR corrected later
                adjusted_p_value = p_value

                # Combined score (similar to Enrichr's approach)
                combined_score = -np.log10(p_value) * np.log2(overlap_count + 1)

                # Create standardized result
                result_dict = create_standardized_results(
                    cluster_name=cluster_name,
                    cluster_genes=cluster_genes,
                    cluster_seed_genes=cluster_seed_genes,
                    cluster_type=self.cluster_type,
                    total_genes_analyzed=overlap_count,
                    percentage=percentage,
                    mean_statistic=combined_score,
                    p_value=p_value,
                    adjusted_p_value=adjusted_p_value,
                    statistic_name='combined_score',
                    genes_in_expression_data=overlap_count
                )
                result_dict['phenotypic_series'] = ps_title
                result_dict['ps_code'] = ps_code
                result_dict['genes_in_ps'] = len(ps_genes_set)
                result_dict['overlapping_genes'] = ';'.join(sorted(overlapping_genes))
                result_dict['overlap_count'] = overlap_count

                summarized_results.append(result_dict)
                results_added += 1

            print(f"  ✓ Found {results_added} phenotypic series overlaps")

        print(f"\n{'='*80}")
        print(f"ENRICHMENT ANALYSIS COMPLETE")
        print(f"{'='*80}")
        print(f"Total phenotypic series enrichments across all clusters: {len(summarized_results)}")

        results_df = pd.DataFrame(summarized_results)

        print(f"\nResults DataFrame shape: {results_df.shape}")

        # Apply Bonferroni correction FIRST (before gene-level results)
        if not results_df.empty and len(results_df) > 1:
            print(f"\nApplying Bonferroni multiple testing correction...")
            _, adjusted_pvalues, _, _ = multipletests(results_df['p_value'].values, method='bonferroni')
            results_df['adjusted_p_value'] = adjusted_pvalues
            sig_count = (adjusted_pvalues < 0.05).sum()
            print(f"  Significant phenotypic series (FDR < 0.05): {sig_count}/{len(results_df)}")
            
        if not results_df.empty:
            # Show distribution of phenotypic series per cluster
            ps_per_cluster = results_df.groupby('cluster_id').size()
            print(f"\nPhenotypic series per cluster statistics:")
            print(f"  Mean: {ps_per_cluster.mean():.1f}")
            print(f"  Median: {ps_per_cluster.median():.1f}")
            print(f"  Max: {ps_per_cluster.max()}")
            print(f"  Min: {ps_per_cluster.min()}")

        # NOW create gene-level results (after FDR correction)
        gene_level_results = []

        for cluster_idx, (cluster_name, cluster_genes) in enumerate(self.clusters.items()):
            cluster_genes_list = [str(g).upper() for g in cluster_genes]
            cluster_seed_genes = self.cluster_seed_mapping[cluster_name]

            # Get best result for this cluster for gene-level annotation
            cluster_results = results_df[results_df['cluster_id'] == cluster_name]
            if not cluster_results.empty:
                best_result = cluster_results.loc[cluster_results['mean_combined_score'].idxmax()]
                best_ps = best_result['phenotypic_series']
                best_p_value = best_result['p_value']
                best_adjusted_p_value = best_result['adjusted_p_value']
                best_combined_score = best_result['mean_combined_score']

                # Create gene-level results
                for gene in cluster_genes_list:
                    is_seed = gene in cluster_seed_genes

                    gene_level_results.append({
                        'cluster_id': cluster_name,
                        'gene': gene,
                        'is_seed_gene': is_seed,
                        'is_in_expression_data': True,
                        'cluster_combined_score': best_combined_score,
                        'cluster_p_value': best_p_value,
                        'cluster_adjusted_p_value': best_adjusted_p_value,
                        'cluster_top_ps': best_ps
                    })

        gene_results_df = pd.DataFrame(gene_level_results)

        print(f"Gene Results DataFrame shape: {gene_results_df.shape}")

        if not gene_results_df.empty:
            print(f"\nAnnotating gene results with cluster significance...")
            cluster_has_sig_ps = results_df[results_df['adjusted_p_value'] < 0.05]['cluster_id'].unique()
            gene_results_df['is_cluster_significant'] = gene_results_df['cluster_id'].isin(cluster_has_sig_ps)
            sig_count = gene_results_df['is_cluster_significant'].sum()
            print(f"  {sig_count} genes in clusters with significant phenotypic series enrichments")

        self.results = results_df
        self.gene_results = gene_results_df

        print(f"\n{'='*80}\n")

        return results_df, gene_results_df
    
    def create_visualizations(self):
        """Create standardized visualizations."""
        print(f"\n{'='*80}")
        print(f"CREATING VISUALIZATIONS")
        print(f"{'='*80}")
        
        plots_dir = os.path.join('plots', self.ppi_source, f'omim_ps_analysis_{self.cluster_type}_{self.safe_prefix}')
        ensure_dir(plots_dir)
        print(f"Plots directory: {plots_dir}")
        
        if self.results.empty:
            print("✗ No results to visualize")
            return
        
        print(f"Creating visualizations for {len(self.results)} results...")
        
        # 1. Volcano plot
        print("\n1. Creating volcano plot...")
        plt.figure(figsize=(10, 8))
        
        self.results['neg_log_pvalue'] = -np.log10(self.results['p_value'] + 1e-300)
        self.results['log2_combined_score'] = self.results['mean_combined_score'].apply(self.safe_log2)
        
        colors = ['red' if pval < 0.05 else 'gray' for pval in self.results['adjusted_p_value']]
        sig_count = sum(1 for c in colors if c == 'red')
        print(f"   Significant results (red): {sig_count}/{len(colors)}")
        
        plt.scatter(self.results['log2_combined_score'], self.results['neg_log_pvalue'], 
                   c=colors, alpha=0.6, s=100)
        
        plt.axhline(y=-np.log10(0.05), color='black', linestyle='--', alpha=0.5)
        
        plt.xlabel('log2(Combined Score)')
        plt.ylabel('-log10(p-value)')
        plt.title(f'OMIM Phenotypic Series Analysis - {self.ppi_source}')
        plt.tight_layout()
        
        volcano_path = os.path.join(plots_dir, 'omim_ps_volcano_plot.png')
        plt.savefig(volcano_path, dpi=300, bbox_inches='tight')
        plt.close()
        print(f"   ✓ Saved: {volcano_path}")
        
        # 2. Top enrichments bar plot
        print("\n2. Creating top enrichments bar plot...")
        if not self.results.empty:
            top_results = self.results.nlargest(20, 'mean_combined_score')
            print(f"   Plotting top {len(top_results)} enrichments")
            
            plt.figure(figsize=(14, 10))
            y_labels = [f"{row['cluster_id'][:30]}: {row['phenotypic_series'][:50]}" 
                       for _, row in top_results.iterrows()]
            plt.barh(range(len(top_results)), top_results['mean_combined_score'])
            plt.yticks(range(len(top_results)), y_labels, fontsize=8)
            plt.xlabel('Combined Score')
            plt.title(f'Top 20 OMIM Phenotypic Series Enrichments - {self.ppi_source}')
            plt.tight_layout()
            
            bar_path = os.path.join(plots_dir, 'top_ps_enrichments.png')
            plt.savefig(bar_path, dpi=300, bbox_inches='tight')
            plt.close()
            print(f"   ✓ Saved: {bar_path}")
        
        print(f"\n{'='*80}\n")
    
    def generate_report(self):
        """Generate a simple text report."""
        print(f"\n{'='*80}")
        print(f"GENERATING REPORT")
        print(f"{'='*80}")
        
        report_file = os.path.join(self.analysis_dir, 'analysis_report.txt')
        print(f"Report file: {report_file}")
        
        with open(report_file, 'w') as f:
            f.write(f"OMIM Phenotypic Series Analysis Report\n")
            f.write(f"======================================\n\n")
            f.write(f"PPI Source: {self.ppi_source}\n")
            f.write(f"Cluster Type: {self.cluster_type}\n")
            f.write(f"Annotation: {self.safe_prefix}\n")
            f.write(f"OMIM PS File: {self.omim_ps_file}\n")
            f.write(f"Total clusters: {len(self.clusters)}\n\n")
            
            if not self.results.empty:
                f.write(f"Analysis Results:\n")
                f.write(f"Total phenotypic series enrichments: {len(self.results)}\n")
                f.write(f"Significant enrichments (adj p < 0.05): {(self.results['adjusted_p_value'] < 0.05).sum()}\n")
                
                # PS per cluster
                ps_per_cluster = self.results.groupby('cluster_id').size()
                f.write(f"\nPhenotypic series per cluster:\n")
                f.write(f"  Mean: {ps_per_cluster.mean():.1f}\n")
                f.write(f"  Median: {ps_per_cluster.median():.1f}\n")
                f.write(f"  Range: {ps_per_cluster.min()}-{ps_per_cluster.max()}\n\n")
                
                # Top enrichments
                f.write("Top 20 Enrichments:\n")
                top_results = self.results.nlargest(20, 'mean_combined_score')
                for idx, row in top_results.iterrows():
                    f.write(f"\n  {row['cluster_id']}: {row['phenotypic_series']} ({row['ps_code']})\n")
                    f.write(f"    Combined Score={row['mean_combined_score']:.2f}, ")
                    f.write(f"p={row['p_value']:.2e}, adj_p={row['adjusted_p_value']:.2e}\n")
                    f.write(f"    Overlap: {row['overlap_count']}/{len(self.clusters[row['cluster_id']])} genes\n")
        
        print(f"✓ Report saved: {report_file}")
        print(f"\n{'='*80}\n")
    
    def run_analysis(self):
        """Run the complete analysis pipeline."""
        print(f"\n{'#'*80}")
        print(f"# STARTING OMIM PHENOTYPIC SERIES ANALYSIS")
        print(f"# PPI Source: {self.ppi_source}")
        print(f"{'#'*80}\n")
        
        # Load clusters
        print(f"STEP 1: Loading HiDeF clusters from comparison results...")
        if not self.load_hidef_clusters_multiplex():
            print("✗ Failed to load HiDeF clusters")
            return False

        print(f"✓ Loaded {len(self.clusters)} clusters with {len(self.seed_genes)} seed genes")
        print(f"  Cluster names: {list(self.clusters.keys())[:5]}{'...' if len(self.clusters) > 5 else ''}")
       
        # Run enrichment analysis
        print(f"\nSTEP 2: Running phenotypic series enrichment analysis...")
        results_df, gene_results_df = self.analyze_enrichment()
        
        # Save results
        if not results_df.empty:
            print(f"\nSTEP 3: Saving results...")
            save_results(results_df, gene_results_df, self.analysis_dir, 'omim_ps')
            print(f"✓ Results saved to: {self.analysis_dir}")
            
            # Create visualizations
            print(f"\nSTEP 4: Creating visualizations...")
            self.create_visualizations()
            
            # Generate report
            print(f"\nSTEP 5: Generating report...")
            self.generate_report()
            
            print(f"\n{'#'*80}")
            print(f"# ANALYSIS COMPLETE!")
            print(f"# Results saved to: {self.analysis_dir}")
            print(f"{'#'*80}\n")
            
            return True
        else:
            print("\n✗ No results generated")
            print(f"  - Clusters loaded: {len(self.clusters) if self.clusters else 0}")
            print(f"  - Results DataFrame empty: {results_df.empty}")
            print(f"  - Gene Results DataFrame empty: {gene_results_df.empty}")
            return False




In [3]:
pwd()

'/lisc/home/user/gerardursin/multiplex/phen'

In [4]:
# Usage:
# analyzer = OMIMPhenotypicSeriesAnalyzer(
#     ppi_source='combined',
#     input_dir='../output',
#     output_dir='./output',
#     omim_ps_file='omim_phenotypic_series_enrichment.csv',
#     cluster_type='disease_module_hidef',
#     annotation_prefix=None
# )
# analyzer.run_analysis()

ppi_source='combined26_multiplex'
input_dir='./output'
output_dir='./output'
persistence_threshold=None

analyzer = OMIMPhenotypicSeriesAnalyzer(
    ppi_source=ppi_source,
    input_dir=input_dir,
    output_dir=output_dir,     
    omim_ps_file='../../phenotypic_series/omim_phenotypic_series_enrichment.csv',
    persistence_threshold=persistence_threshold
)
analyzer.run_analysis()


INITIALIZING OMIMPhenotypicSeriesAnalyzer
PPI Source: combined26_multiplex
Input Dir: ./output
Output Dir: ./output
OMIM PS File: ../../phenotypic_series/omim_phenotypic_series_enrichment.csv

Loading OMIM Phenotypic Series data...
  ✓ Loaded 5464 phenotypic series entries
  Columns: ['ps_code', 'ps_title', 'location', 'phenotype', 'inheritance', 'phenotype_mim', 'genes', 'gene_mim']

Output Configuration:
  Analysis directory: ./output/combined26_multiplex/omim_ps_analysis_hidef_from_comparison
  Directory created/verified: True


################################################################################
# STARTING OMIM PHENOTYPIC SERIES ANALYSIS
# PPI Source: combined26_multiplex
################################################################################

STEP 1: Loading HiDeF clusters from comparison results...
load_multiplex_clusters(/lisc/home/user/gerardursin/multiplex/11_hidef_module/results_cpm/heterogeneous_hidef_faithful_cpm.nodes): kept 2187 clusters (persistence

True